human lung cancer

导入相关包

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
from sklearn.metrics import (homogeneity_score,v_measure_score,adjusted_mutual_info_score,normalized_mutual_info_score,adjusted_rand_score,fowlkes_mallows_score)


In [2]:
repeat_times = 2
simple_path = '/home/cavin/jt/benchmark/data/human_lung_cancer/SMI_Lung.h5ad'
celltype_col = 'cell_type'
cell_emb_col = 'X_scGPT'
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations/scgpt_human_lung_cancer_emb.parquet"


读取嵌入

In [3]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [4]:
adata.obsm[cell_emb_col] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial', 'X_scGPT'

In [5]:
res_list = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0]
random_seed=2026 + repeat_times * 200
key_added='leiden'
sc.pp.neighbors(adata, use_rep=cell_emb_col,random_state=random_seed)
adata


AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    uns: 'neighbors'
    obsm: 'spatial', 'X_scGPT'
    obsp: 'distances', 'connectivities'

In [6]:
max_value = 0
metrics = {"method": "scGPT", "ARI": 0, "NMI": 0, "AMI": 0, "HS": 0, "VM": 0, "FMI": 0, "mean value": 0}
save_label_df = None

In [7]:

for used_res in res_list:
    sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)
    true_labels = np.array(adata.obs[celltype_col])
    cluster_labels = np.array(adata.obs[key_added])
    FMI = fowlkes_mallows_score(true_labels, cluster_labels)
    homogeneity = homogeneity_score(true_labels, cluster_labels)
    v_measure = v_measure_score(true_labels, cluster_labels)
    ami = adjusted_mutual_info_score(true_labels, cluster_labels)
    nmi = normalized_mutual_info_score(true_labels, cluster_labels)
    ari = adjusted_rand_score(true_labels, cluster_labels)
    mean_value = np.mean([FMI, homogeneity, v_measure, ami, nmi, ari])

    n_cluster = len(adata.obs[key_added].unique())
    n_celltype = len(adata.obs[celltype_col].unique())
    if mean_value > max_value:
        save_label_df = adata.obs[key_added]
        metrics["ARI"] = ari
        metrics["NMI"] = nmi
        metrics["AMI"] = ami
        metrics["HS"] = homogeneity
        metrics["VM"] = v_measure
        metrics["FMI"] = FMI
        metrics["mean value"] = mean_value
        max_value = mean_value

    print(f'resolution = {used_res} | n_celltype = {n_celltype} | n_cluster = {n_cluster} | mean_value = {round(mean_value,3)} | ARI = {round(ari,3)} |  NMI = {round(nmi,3)} | AMI = {round(ami,3)} | HS = {round(homogeneity,3)} | VM = {round(v_measure,3)} | FMI = {round(FMI,3)} ')

/tmp/ipykernel_2626619/737231793.py:2: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, random_state=random_seed, resolution=used_res,key_added=key_added)


resolution = 0.1 | n_celltype = 18 | n_cluster = 2 | mean_value = 0.362 | ARI = 0.301 |  NMI = 0.355 | AMI = 0.355 | HS = 0.237 | VM = 0.355 | FMI = 0.57 
resolution = 0.2 | n_celltype = 18 | n_cluster = 2 | mean_value = 0.363 | ARI = 0.304 |  NMI = 0.355 | AMI = 0.354 | HS = 0.237 | VM = 0.355 | FMI = 0.572 
resolution = 0.3 | n_celltype = 18 | n_cluster = 3 | mean_value = 0.397 | ARI = 0.431 |  NMI = 0.355 | AMI = 0.355 | HS = 0.269 | VM = 0.355 | FMI = 0.613 
resolution = 0.4 | n_celltype = 18 | n_cluster = 4 | mean_value = 0.466 | ARI = 0.526 |  NMI = 0.422 | AMI = 0.422 | HS = 0.336 | VM = 0.422 | FMI = 0.667 
resolution = 0.5 | n_celltype = 18 | n_cluster = 7 | mean_value = 0.414 | ARI = 0.395 |  NMI = 0.397 | AMI = 0.396 | HS = 0.365 | VM = 0.397 | FMI = 0.534 
resolution = 0.6 | n_celltype = 18 | n_cluster = 8 | mean_value = 0.343 | ARI = 0.234 |  NMI = 0.36 | AMI = 0.359 | HS = 0.358 | VM = 0.36 | FMI = 0.386 
resolution = 0.7 | n_celltype = 18 | n_cluster = 9 | mean_value = 0

In [8]:
save_label_df

unique_ID
1-2        1
1-3        1
1-4        2
1-6        1
1-7        1
          ..
20-4048    0
20-4049    0
20-4050    0
20-4051    0
20-4052    0
Name: leiden, Length: 87589, dtype: category
Categories (4, object): ['0', '1', '2', '3']

In [9]:
save_label_df_path = f"/home/cavin/jt/benchmark/experiments/results/labels_df/lung_cancer/scgpt_human_lung_cancer_labels_repeat_{repeat_times}.csv"

In [10]:
save_label_df.to_csv(save_label_df_path)

In [11]:
metrics

{'method': 'scGPT',
 'ARI': 0.5264308172112916,
 'NMI': 0.42198565727360476,
 'AMI': 0.4218798776638462,
 'HS': 0.33639862716909,
 'VM': 0.4219856572736048,
 'FMI': 0.6671828832378751,
 'mean value': 0.4659772533048854}

In [12]:
metrics_df = pd.DataFrame.from_dict(metrics, orient='index').T

In [13]:
metrics_df_save_path = f"/home/cavin/jt/benchmark/experiments/results/cluster_metrics/lung_cancer/human_lung_cancer_metrics_repeat_{repeat_times}.csv"

In [14]:
metrics_df.to_csv(metrics_df_save_path, index=False,mode="a", header=not pd.io.common.file_exists(metrics_df_save_path))

In [2]:
import pandas as pd
df0 = pd.read_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/lung_cancer/human_lung_cancer_metrics_repeat_0.csv",header=0,index_col=0)
df1 = pd.read_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/lung_cancer/human_lung_cancer_metrics_repeat_1.csv",header=0,index_col=0)
df2 = pd.read_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/lung_cancer/human_lung_cancer_metrics_repeat_2.csv",header=0,index_col=0)

In [3]:
df0

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
scGPT,0.525147,0.401014,0.400873,0.331506,0.401014,0.656915,0.452745
scFoundation,0.374047,0.398514,0.398235,0.418436,0.398514,0.503624,0.415229
OmniCell,0.355218,0.330571,0.330359,0.309205,0.330571,0.495465,0.358565
Nicheformer,0.145675,0.331568,0.331251,0.341598,0.331568,0.307561,0.298203
Geneformer,0.077629,0.046613,0.046544,0.031180,0.046613,0.410304,0.109814
PCA,0.446384,0.478934,0.478837,0.375037,0.478934,0.619299,0.479571


In [4]:
df1

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
scGPT,0.527054,0.405901,0.405826,0.313044,0.405901,0.671435,0.454860
OmniCell,0.290825,0.344080,0.343772,0.360059,0.344080,0.430600,0.352236
Nicheformer,0.216967,0.339991,0.339809,0.243853,0.339991,0.496676,0.329548
Geneformer,0.074747,0.045709,0.045639,0.030597,0.045709,0.407605,0.108334
scFoundation,0.332476,0.392929,0.392649,0.415245,0.392929,0.467263,0.398915
PCA,0.269344,0.468116,0.467996,0.403450,0.468116,0.453508,0.421755


In [5]:
df2

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
scGPT,0.526431,0.421986,0.421880,0.336399,0.421986,0.667183,0.465977
OmniCell,0.330885,0.349848,0.349591,0.348192,0.349848,0.467939,0.366051
Nicheformer,0.205956,0.345417,0.345133,0.351270,0.345417,0.359000,0.325365
Geneformer,0.075409,0.045855,0.045786,0.030688,0.045855,0.408276,0.108645
scFoundation,0.391168,0.388342,0.388077,0.392750,0.388342,0.519638,0.411386
PCA,0.427402,0.465348,0.465247,0.361210,0.465348,0.609816,0.465728


In [6]:
df = df0+df1+df2

In [7]:
df

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
Geneformer,0.227785,0.138177,0.137969,0.092465,0.138177,1.226185,0.326793
Nicheformer,0.568598,1.016976,1.016193,0.936721,1.016976,1.163236,0.953117
OmniCell,0.976928,1.024498,1.023722,1.017456,1.024498,1.394005,1.076851
PCA,1.143131,1.412398,1.412080,1.139698,1.412398,1.682623,1.367055
scFoundation,1.097691,1.179786,1.178962,1.226431,1.179786,1.490525,1.225530
scGPT,1.578632,1.228901,1.228579,0.980948,1.228901,1.995533,1.373582


In [8]:
df = df / 3

In [9]:
df

,ARI,NMI,AMI,HS,VM,FMI,mean value
method,,,,,,,
Geneformer,0.075928,0.046059,0.045990,0.030822,0.046059,0.408728,0.108931
Nicheformer,0.189533,0.338992,0.338731,0.312240,0.338992,0.387745,0.317706
OmniCell,0.325643,0.341499,0.341241,0.339152,0.341499,0.464668,0.358950
PCA,0.381044,0.470799,0.470693,0.379899,0.470799,0.560874,0.455685
scFoundation,0.365897,0.393262,0.392987,0.408810,0.393262,0.496842,0.408510
scGPT,0.526211,0.409634,0.409526,0.326983,0.409634,0.665178,0.457861


In [10]:
df.to_csv("/home/cavin/jt/benchmark/experiments/results/cluster_metrics/lung_cancer/human_lung_cancer_metrics_mean.csv")